## **COMETA medieval corpus — sample categorization**

Scan `data/raw/COMETA_medieval_corpus/*.txt` (real Old Occitan transcriptions) and label every line that contains patterns our HTR model currently struggles with:

- **`am`** — Old Occitan preposition "with". Whole-word match.
- **`ma`** — possessive "my" / noun "hand" / inflected verb forms. Whole-word match.
- **`roman_numeral`** — strict valid Roman numeral, with medieval allowances (`IIII`, trailing `j`). Dotted forms (`.III.`, `.X.`) accepted at any length; undotted forms require ≥3 characters to exclude Occitan words `mi`/`li`/`vi`/`xi` that happen to parse as valid Romans.

Output: `data/processed/synthetic_seeds/cometa_categorized.json` — a dictionary keyed by `filename:lineno` with the categories and raw text per line, plus a summary block. Used downstream as the source of training lines for synthetic image generation.

In [1]:
import json
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])

CORPUS_DIR = project_root / "data/raw/COMETA_medieval_corpus"
OUT_PATH   = project_root / "data/processed/synthetic_seeds/cometa_categorized.json"

print(f"corpus:  {CORPUS_DIR}")
print(f"output:  {OUT_PATH}")

corpus:  /Users/camilabermudezvalderrama/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/raw/COMETA_medieval_corpus
output:  /Users/camilabermudezvalderrama/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/synthetic_seeds/cometa_categorized.json


### Pattern definitions

Tune these here if the criteria don't match what you want. The `has_roman_numeral` function is the tricky one — see its docstring for the rationale on length / dot-bracketing rules.

In [2]:
# Strict Roman numeral validator. Medieval allowances: IIII permitted
# (M{0,4} etc. allow up to 4 of each); trailing 'j' is the medieval variant
# of final 'i' (iij, iiij, vij, viij).
RE_ROMAN_VALID = re.compile(
    r"^m{0,4}(cm|cd|d?c{0,4})(xc|xl|l?x{0,4})(ix|iv|v?i{0,4})j?$",
    re.IGNORECASE,
)

# Candidate Roman tokens: pure [ivxlcdm] (with optional trailing j), with
# capture groups for surrounding dots so we can require dot-bracketing for
# short forms.
RE_CANDIDATE = re.compile(
    r"(?<![A-Za-z])(\.?)([ivxlcdm]+j?)(\.?)(?![A-Za-z])",
    re.IGNORECASE,
)

RE_AM = re.compile(r"\bam\b", re.IGNORECASE)
RE_MA = re.compile(r"\bma\b", re.IGNORECASE)


def has_roman_numeral(line: str) -> bool:
    """True if the line contains at least one token that strict-parses as a
    Roman numeral.

    Dotted forms (e.g. `.III.`, `.X.`) are accepted at any length because
    the dots are a strong manuscript convention for "this is a numeral."
    Undotted forms require ≥3 characters to exclude the Occitan words
    `mi` / `li` / `vi` / `xi` — all of which parse as valid Romans
    (1001 / 51 / 6 / 11) but are ordinary vocabulary in context.

    One residual false positive: undotted `dix` (valid Roman 509) is also
    an Occitan verb form (`dire` → `dix`). Blocklist it explicitly if it
    contaminates downstream training.
    """
    for m in RE_CANDIDATE.finditer(line):
        pre, tok, post = m.group(1), m.group(2), m.group(3)
        dotted = bool(pre) or bool(post)
        if not dotted and len(tok) < 3:
            continue
        if not RE_ROMAN_VALID.match(tok):
            continue
        return True
    return False

### Scan

Walk every file, classify every non-empty line, build the master dict.

In [3]:
samples: dict[str, dict] = {}            # "file:line" -> {categories, text}
per_cat_count: Counter = Counter()
per_cat_files: dict[str, set] = defaultdict(set)

files = sorted(CORPUS_DIR.glob("*.txt"))

for path in files:
    for i, raw in enumerate(
        path.read_text(encoding="utf-8", errors="replace").splitlines(), start=1
    ):
        line = raw.strip()
        if not line:
            continue
        cats: list[str] = []
        if RE_AM.search(line):       cats.append("am")
        if RE_MA.search(line):       cats.append("ma")
        if has_roman_numeral(line):  cats.append("roman_numeral")
        if not cats:
            continue
        samples[f"{path.name}:{i}"] = {"categories": cats, "text": line}
        for c in cats:
            per_cat_count[c] += 1
            per_cat_files[c].add(path.name)

multi_count = sum(1 for v in samples.values() if len(v["categories"]) > 1)
print(f"Scanned {len(files)} files; {len(samples):,} lines have ≥1 category; "
      f"{multi_count:,} cover more than one.")

Scanned 17 files; 8,495 lines have ≥1 category; 370 cover more than one.


### Category summary

In [4]:
summary_rows = [
    {
        "category": c,
        "n_lines":  per_cat_count[c],
        "n_files":  len(per_cat_files[c]),
    }
    for c in ("am", "ma", "roman_numeral")
]
pd.DataFrame(summary_rows)

,category,n_lines,n_files
0,am,2940,16
1,ma,1215,16
2,roman_numeral,4730,16


### Preview — first 6 examples per category

Sanity-check that the regex is doing what you expect. If you see lines that obviously don't contain the pattern, the regex needs tightening; if you don't see lines you'd expect, it needs loosening.

In [5]:
N_PREVIEW = 6
preview_rows = []
for c in ("am", "ma", "roman_numeral"):
    seen = 0
    for sid, info in samples.items():
        if c in info["categories"]:
            preview_rows.append({
                "category":  c,
                "sample_id": sid,
                "text":      info["text"][:120],
            })
            seen += 1
            if seen >= N_PREVIEW:
                break
pd.DataFrame(preview_rows)

,category,sample_id,text
0,am,Additional_10323.txt:5,E olptrant e planhent am lagremas et am plors
1,am,Additional_10323.txt:131,Cayrar d aquesta flamma am bella clamor la
2,am,Additional_10323.txt:206,Car moron crestians. am tota lur companha
3,am,Additional_10323.txt:231,Es am de mayntas colors.
4,am,Additional_10323.txt:233,Cavalta am gran companhia
5,am,Additional_10323.txt:268,.III. homes am mot fer belaure
6,ma,Additional_10323.txt:28,A ma dona maria
7,ma,Additional_10323.txt:57,Complisca ma defauta
8,ma,Additional_10323.txt:63,Car ma lengna non es
9,ma,Additional_10323.txt:297,Cassaras e gitaras ma


### Multi-category lines — denser training signal

Lines covering more than one category are especially useful for synthesis: one rendered image teaches the model multiple confusable patterns at once.

In [ ]:
multi_rows = []
for sid, info in samples.items():
    if len(info["categories"]) > 1:
        multi_rows.append({
            "sample_id":  sid,
            "categories": info["categories"],
            "text":       info["text"][:120],
        })
        if len(multi_rows) >= 10:
            break
print(f"{multi_count:,} multi-category lines total. First 10:")
pd.DataFrame(multi_rows)

### Save

In [7]:
doc = {
    "criteria": {
        "am": r"\bam\b (case-insensitive, whole-word)",
        "ma": r"\bma\b (case-insensitive, whole-word)",
        "roman_numeral": (
            "strict valid Roman numeral (medieval IIII allowed, trailing 'j' allowed). "
            "Dotted forms accepted at any length; undotted forms require ≥3 chars to "
            "exclude Occitan words 'mi'/'li'/'vi'/'xi'. Known residual false positive: "
            "undotted 'dix' (valid Roman 509, also Occitan verb form)."
        ),
    },
    "summary": {
        "total_files_scanned":                len(files),
        "total_lines_with_at_least_one_category": len(samples),
        "lines_per_category":                 {c: per_cat_count[c] for c in ("am", "ma", "roman_numeral")},
        "files_per_category":                 {c: sorted(per_cat_files[c]) for c in ("am", "ma", "roman_numeral")},
        "multi_category_lines":               multi_count,
    },
    "samples": samples,
}

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(json.dumps(doc, indent=2, ensure_ascii=False))
print(f"Saved {len(samples):,} samples → {OUT_PATH}")

Saved 8,495 samples → /Users/camilabermudezvalderrama/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/data/processed/synthetic_seeds/cometa_categorized.json


### Downstream use — load and filter

This cell is self-contained (re-loads the saved JSON), so you can run it independently from a fresh kernel without re-scanning the corpus.

In [8]:
import random

with open(OUT_PATH) as f:
    doc = json.load(f)

# All lines containing "am" as a word.
am_lines = [s["text"] for s in doc["samples"].values() if "am" in s["categories"]]

# All lines containing BOTH "am" and a Roman numeral — densest training signal.
am_and_roman = [
    s["text"] for s in doc["samples"].values()
    if "am" in s["categories"] and "roman_numeral" in s["categories"]
]

print(f"am          : {len(am_lines):>5,} lines")
print(f"am + roman  : {len(am_and_roman):>5,} lines")

# A reproducible random sample to hand to a synthetic renderer.
random.seed(0)
sample_for_render = random.sample(am_lines, k=min(20, len(am_lines)))
print("\n20 random 'am' lines for rendering:")
for ln in sample_for_render[:5]:
    print(f"  • {ln[:100]}")

am          : 2,940 lines
am + roman  :   133 lines

20 random 'am' lines for rendering:
  • matinas. despendia am gran aondan-
  • e dizia a la maire am gran compa-
  • Viron estar la donna am son gent cors entier
  • En savista am nom propri
  • am mot gran compagnia.
